[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/15_django/lab02_serving_a_model_with_auth.ipynb)

# 🧪 Lab 2 — Serving a churn model with Django: API, auth & history

> **Module:** Django for AI Web Apps (Module 15) · **Estimated time:** ~60 minutes · **Difficulty:** Intermediate → Advanced

[Lab 1](lab01_django_in_a_notebook.ipynb) rebuilt Django's engine room — settings, models, the ORM, templates, views, forms — one piece at a time. This lab ships the product. We train a **real scikit-learn churn model**, dump it to disk with `joblib`, load it **once** behind a Django app, and serve it two ways: a **JSON API** for robots and a **validated HTML form** for humans — with every single prediction logged to the database, and the prediction log locked behind a **login**. That is the complete **ChurnScope** pattern from [serving-a-model.md](serving-a-model.md), [auth-and-history.md](auth-and-history.md) and [deployment.md](deployment.md) — running live in this kernel, no server, no browser.

> 🧭 **Mental model: an artifact behind an endpoint.** A served model is four verbs, in strict order —
> **train offline → ship a file → load once at startup → predict per request** (and *log every prediction*).
> Training happens on your schedule, in a batch job. What crosses into the web app is a *file*. The app pays
> the loading cost once, at boot, and every request after that finds a warm model, spends milliseconds, and
> leaves a `Prediction` row behind as its receipt. Every section of this lab is one of those verbs.

**Prerequisites.** [Lab 1](lab01_django_in_a_notebook.ipynb) explains the notebook-boot trick slowly (settings, app registration, string templates) — this lab replays it in *one condensed cell* and moves on, so skim Lab 1 first if the setup cell looks like sorcery. NB 14 (scikit-learn pipelines) and NB 39 (from notebook to project) are helpful but not required. Django is the only extra dependency — §1 checks. **Optional module**: skip or skim guilt-free.

## 🎯 Learning objectives

By the end of this lab you can:

1. Train a scikit-learn churn **pipeline** offline, ship it as a **versioned `joblib` artifact**, and load it **once at startup** — never per request.
2. Serve predictions through a **JSON API** (`POST /api/score/`) that validates input, returns clean errors (400/405), and logs every score through the ORM.
3. Serve the *same* model through a **validated HTML form**, and explain what `@csrf_exempt` disables, why the JSON API needs it, and what production uses instead.
4. Protect a page with **`@login_required`**, create users safely with `create_user`, and walk the **302 → login → 200** flow with the test client.
5. Compress the whole app into a **test suite** — status codes, probability bounds, auth boundaries — that runs in milliseconds without a server.
6. Audit the app against a **production checklist** (DEBUG, SECRET_KEY, ALLOWED_HOSTS, gunicorn, collectstatic) and read the example app's real **Dockerfile** line by line.

## 1. Smoke test — is the toolbox complete?

Two toolboxes, actually. The **modelling** side — scikit-learn, joblib, pandas, numpy — is core course equipment (NB 14 onwards) and preinstalled on Colab. The **web** side is Django, this module's one extra dependency (`pip install django`, or uncomment the `%pip` line on Colab). As in Lab 1, every Django cell checks `HAS_DJANGO` and prints a skip note instead of crashing — the *modelling* cells run either way.

In [ ]:
# On Colab or a fresh machine: uncomment the next line, run it, then restart the kernel.
# %pip install django

try:
    import django
    HAS_DJANGO = True
    print(f"Django {django.get_version()} — ready.")
except ImportError:
    HAS_DJANGO = False
    print("Django not installed — install with:  pip install django")
    print("Django cells below will print a skip note; the modelling cells still run.")

import joblib
import numpy as np
import pandas as pd
import sklearn

print(f"scikit-learn {sklearn.__version__} · pandas {pd.__version__} · joblib ready")

## 2. Boot ChurnScope's engine — Lab 1, condensed to one cell

Everything Lab 1 built across four sections, restored in one breath: configure settings by hand, register a `scoring` app (an app is just an importable package — we write a two-file stub into a temp folder), point the database at a fresh SQLite file, and `migrate`. If any line looks mysterious, [Lab 1](lab01_django_in_a_notebook.ipynb) §2 walks it slowly — including the 🔬 story behind `DJANGO_ALLOW_ASYNC_UNSAFE` (Jupyter's kernel runs an asyncio event loop, and Django's ORM refuses to make blocking database calls inside one unless you set that flag; it's the standard notebook escape hatch).

Three things are **new** relative to Lab 1's settings, and all three exist because *this* lab has users:

| Setting | Lab 1 | This lab | Why |
|---|---|---|---|
| `INSTALLED_APPS` | contenttypes, auth, `scoring` | + **`django.contrib.sessions`** | a login *is* a session — it needs a table to live in |
| `MIDDLEWARE` | *(none needed)* | **Session → Csrf → Authentication** | sessions attach the cookie, CSRF guards forms, auth turns the session into `request.user` |
| auth redirects | — | `LOGIN_URL`, `LOGIN_REDIRECT_URL`, `LOGOUT_REDIRECT_URL` | the three signposts from [auth-and-history.md](auth-and-history.md) |

One more small addition: the `auth` **context processor**, so templates automatically receive the current `user` — a convenience `startproject`'s default settings include and our hand-rolled `TEMPLATES` must opt into.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

if HAS_DJANGO:
    from django.conf import settings

    # Jupyter runs an asyncio event loop; without this flag every ORM call would
    # raise SynchronousOnlyOperation. Standard notebook fix — Lab 1 §2 has the 🔬.
    os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

    LAB_DIR = Path(tempfile.gettempdir()) / "churnscope_lab2"
    DB_PATH = LAB_DIR / "churnscope_lab2.sqlite3"
    # One dict = the whole template store (locmem loader). Survives re-runs:
    TEMPLATE_REGISTRY = globals().get("TEMPLATE_REGISTRY", {})

    if settings.configured:
        print("Settings already configured — restart the kernel to reconfigure.")
    else:
        # The 'scoring' app: an importable two-file package (Lab 1's trick).
        (LAB_DIR / "scoring").mkdir(parents=True, exist_ok=True)
        (LAB_DIR / "scoring" / "__init__.py").write_text(
            "# The scoring app — same name as ChurnScope's app package.\n")
        (LAB_DIR / "scoring" / "models.py").write_text(
            "# Models are defined in notebook cells instead (see Lab 1 §3).\n")
        sys.path.insert(0, str(LAB_DIR))

        if DB_PATH.exists():                     # fresh database per kernel session
            DB_PATH.unlink()

        settings.configure(
            DEBUG=True,
            SECRET_KEY="lab-only-not-a-real-secret",
            ALLOWED_HOSTS=["testserver"],        # the test client's hostname
            INSTALLED_APPS=[
                "django.contrib.contenttypes",
                "django.contrib.auth",           # User, permissions, login views
                "django.contrib.sessions",       # NEW: logins live in sessions
                "scoring",
            ],
            MIDDLEWARE=[                         # NEW: the auth plumbing, in order
                "django.contrib.sessions.middleware.SessionMiddleware",
                "django.middleware.csrf.CsrfViewMiddleware",
                "django.contrib.auth.middleware.AuthenticationMiddleware",
            ],
            DATABASES={"default": {
                "ENGINE": "django.db.backends.sqlite3",
                "NAME": DB_PATH,
            }},
            ROOT_URLCONF="notebook_urls",        # a module we build in §4
            TEMPLATES=[{
                "BACKEND": "django.template.backends.django.DjangoTemplates",
                "OPTIONS": {
                    "loaders": [
                        ("django.template.loaders.locmem.Loader", TEMPLATE_REGISTRY),
                    ],
                    "context_processors": [      # NEW: hand every template `user`
                        "django.contrib.auth.context_processors.auth",
                    ],
                },
            }],
            # The three auth signposts from auth-and-history.md:
            LOGIN_URL="/accounts/login/",        # where @login_required sends strangers
            LOGIN_REDIRECT_URL="/history/",      # where a fresh login lands
            LOGOUT_REDIRECT_URL="/score/",       # where logout drops you
            USE_TZ=True,
            TIME_ZONE="UTC",
            DEFAULT_AUTO_FIELD="django.db.models.BigAutoField",
        )
        django.setup()                           # populate the app registry — once!

    from django.apps import apps

    print("Installed apps:", [a.label for a in apps.get_app_configs()])
    print("Database file :", settings.DATABASES["default"]["NAME"])
else:
    print("Django not installed — skipping.")

And the furniture from Lab 1's later sections — the **`Prediction` model** and the **`ChurnForm`** — field-for-field from `example-app/scoring/models.py` and `forms.py` (plus the notebook-only `app_label`), followed immediately by `migrate`. The model is our audit trail: what went in, what came out, when.

In [ ]:
if HAS_DJANGO:
    from django import forms
    from django.core.management import call_command
    from django.db import connection, models

    class Prediction(models.Model):
        """One churn score, logged for the dashboard and audit trail."""

        created_at = models.DateTimeField(auto_now_add=True)
        tenure_months = models.PositiveIntegerField()
        monthly_charges = models.FloatField()
        support_tickets = models.PositiveIntegerField()
        contract = models.CharField(max_length=20)
        probability = models.FloatField()
        will_churn = models.BooleanField()

        class Meta:
            app_label = "scoring"        # notebook-only: pin the model to our app
            ordering = ["-created_at"]   # newest first — everywhere, by default

        def __str__(self):
            verdict = "churn" if self.will_churn else "stay"
            return f"{self.contract}: p={self.probability:.2f} ({verdict})"

    CONTRACT_CHOICES = [
        ("month-to-month", "Month-to-month"),
        ("one-year", "One year"),
        ("two-year", "Two year"),
    ]

    class ChurnForm(forms.Form):
        """Validates one customer's details before they reach the scorer."""

        tenure_months = forms.IntegerField(min_value=0, max_value=120, initial=6)
        monthly_charges = forms.FloatField(min_value=0, initial=70.0)
        support_tickets = forms.IntegerField(min_value=0, max_value=50, initial=2)
        contract = forms.ChoiceField(choices=CONTRACT_CHOICES)

    call_command("migrate", run_syncdb=True, verbosity=0)   # ≙ manage.py migrate

    tables = sorted(connection.introspection.table_names())
    print(f"{len(tables)} tables — including:")
    for name in tables:
        if name in ("scoring_prediction", "auth_user", "django_session"):
            print(f"  {name:<20} ← {'ours' if name.startswith('scoring') else 'a battery'}")
else:
    print("Django not installed — skipping.")

## 3. Train offline, ship a file, load once

[serving-a-model.md](serving-a-model.md) ends with a three-step recipe for swapping ChurnScope's transparent stand-in for a real trained model: **(1) train and dump outside Django, (2) load once at startup, (3) keep the signature.** This section performs all three, live.

First, the stand-in itself — `example-app/scoring/scorer.py`, coefficient for coefficient. In the example app it *pretends* to be a model; here we give it a more honest job: it is our **data-generating process**. We'll synthesise a customer table whose churn behaviour follows exactly this logic, train a `LogisticRegression` on it, and check the model *recovers the story* — month-to-month contracts and support tickets push risk up, tenure pushes it down. (With real data you'd skip this step and load your NB 27-style customer table; synthetic-with-known-truth is how you rehearse the machinery.)

In [ ]:
import math

CONTRACT_RISK = {"month-to-month": 0.9, "one-year": 0.0, "two-year": -0.9}


def baseline_churn_probability(*, tenure_months, monthly_charges, support_tickets, contract):
    """The example app's transparent stand-in — here, our ground truth."""
    z = (
        -0.5
        + 1.4 * CONTRACT_RISK.get(contract, 0.0)   # month-to-month is risky
        - 0.05 * float(tenure_months)               # loyalty lowers risk
        + 0.015 * float(monthly_charges)            # pricier plans churn more
        + 0.25 * float(support_tickets)             # friction churns
    )
    return 1.0 / (1.0 + math.exp(-z))


rng = np.random.default_rng(42)
N = 600

customers = pd.DataFrame({
    "tenure_months": rng.integers(0, 73, size=N),
    "monthly_charges": np.round(rng.uniform(20, 120, size=N), 2),
    "support_tickets": rng.poisson(1.5, size=N).clip(max=12),
    "contract": rng.choice(["month-to-month", "one-year", "two-year"],
                           size=N, p=[0.55, 0.25, 0.20]),
})
p_true = customers.apply(lambda row: baseline_churn_probability(**row), axis=1)
customers["churned"] = (rng.random(N) < p_true).astype(int)   # noisy, like real life

print(f"{N} customers · churn rate {customers['churned'].mean():.1%}")
customers.head()

**Step 1 — train and dump, outside Django.** The estimator is a proper **`Pipeline`** (NB 14): a `ColumnTransformer` that standardises the three numeric columns and one-hot encodes `contract`, feeding a `LogisticRegression`. The pipeline *is* the artifact — preprocessing and model travel as one object, so serving code can never apply the wrong scaling.

> ⚠️ **`handle_unknown="ignore"` is a serving decision, made at training time.** The default `OneHotEncoder` *raises* on categories it never saw — which means one customer with `contract="quarterly"` would crash your API at 3 a.m. With `"ignore"` an unseen category encodes as all-zeros: the model quietly falls back to "no contract signal". Decide *how the model should fail* before you ship it, not after.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

FEATURES = ["tenure_months", "monthly_charges", "support_tickets", "contract"]
NUMERIC = ["tenure_months", "monthly_charges", "support_tickets"]

X_train, X_test, y_train, y_test = train_test_split(
    customers[FEATURES], customers["churned"],
    test_size=0.25, random_state=42, stratify=customers["churned"])

pipeline = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("contract", OneHotEncoder(handle_unknown="ignore"), ["contract"]),
    ])),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipeline.fit(X_train, y_train)

auc = roc_auc_score(y_test, pipeline.predict_proba(X_test)[:, 1])
print(f"held-out test set: AUC = {auc:.3f} · accuracy = {pipeline.score(X_test, y_test):.3f}")
print("(labels were sampled with noise, so even the true model couldn't score 1.0)")

Good enough to ship. Now the hand-off: in real life this training script and the web app are **different processes on different schedules** — training is a batch job (cron, CI, Module 11's schedulers); the app just receives a file. `joblib.dump` is the hand-off, and a **version string** rides along, because six months from now "which model produced this score?" will be a real question with real consequences.

> ⚠️ **Version your artifacts.** A model file with no version is a mystery in a trench coat. The chapter's convention — a `MODEL_VERSION` constant next to the artifact it describes, logged with every prediction — is two lines of insurance. (The stretch exercises write it into the `Prediction` table itself.)

In [ ]:
MODEL_VERSION = "2026-07-logreg-v1"          # date + family + revision
MODEL_PATH = LAB_DIR if HAS_DJANGO else Path(tempfile.gettempdir())
MODEL_PATH = MODEL_PATH / "churn_model.joblib"

joblib.dump(pipeline, MODEL_PATH)
print(f"shipped: {MODEL_PATH}")
print(f"         {MODEL_PATH.stat().st_size / 1024:.1f} KB · version {MODEL_VERSION}")

**Step 2 — load once at startup. Step 3 — keep the signature.** The serving side never touches `pipeline` — it loads the *file*, once, into a module-level cache, exactly like the chapter's `scorer.py`. And the public function keeps the stand-in's signature — same name, same keyword-only arguments, same 0-to-1 return — so views, forms and templates can't tell the difference. That signature is **the seam**: everything on one side is ML, everything on the other side is web.

> ⚠️ **Never load — and *never* train — per request.** `joblib.load` in a view means disk I/O plus unpickling on the hot path, thousands of times, for a file that changes only at deploy time; training in a view is that mistake squared. In the real app the one-time load lives in `ScoringConfig.ready()` (`scoring/apps.py`) — Django's "the app just booted" hook. A notebook's equivalent of *startup* is simply: this cell, run once.

In [ ]:
_model = None                       # module-level cache — filled once, read per request


def load_model():
    """Pay the loading cost ONCE. In the real app, ScoringConfig.ready() calls this."""
    global _model
    _model = joblib.load(MODEL_PATH)
    return _model


def churn_probability(*, tenure_months, monthly_charges, support_tickets, contract):
    """Same signature as the stand-in — the seam the whole app talks to.

    The pipeline was trained on a DataFrame with named columns, so serving
    must rebuild that exact schema: same columns, same names, one row.
    """
    row = pd.DataFrame([{
        "tenure_months": int(tenure_months),
        "monthly_charges": float(monthly_charges),
        "support_tickets": int(support_tickets),
        "contract": str(contract),
    }])
    return float(_model.predict_proba(row)[0, 1])    # column 1 = P(churn)


load_model()                        # ← "startup" happens here, exactly once

demo = dict(tenure_months=2, monthly_charges=95.0, support_tickets=5,
            contract="month-to-month")
print(f"trained model : P(churn) = {churn_probability(**demo):.3f}")
print(f"stand-in      : P(churn) = {baseline_churn_probability(**demo):.3f}")
print("→ the LogisticRegression recovered the stand-in's story from noisy labels")

---

### ✋ Quick exercise (~2 min) — Risky vs safe, through the seam

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The example app's test suite pins one business fact: *a month-to-month customer must out-risk an otherwise identical two-year customer.* Verify our trained model kept that promise — score the **same profile** (3 months tenure, 95.0 monthly charges, 4 tickets) under `"month-to-month"` and `"two-year"`, `assert` both probabilities live in `[0, 1]`, and `assert` the risky one is larger. Call only `churn_probability(...)` — the seam, not the pipeline.

In [ ]:
# ✍️ Your turn 👇
profile = dict(tenure_months=3, monthly_charges=95.0, support_tickets=4)

# p_risky = churn_probability(**profile, contract=...)
p_risky = ...
p_safe = ...

# assert 0.0 <= p_risky <= 1.0 and 0.0 <= p_safe <= 1.0
# assert p_risky > p_safe, "month-to-month should out-risk two-year"
# print(f"month-to-month {p_risky:.3f}  >  two-year {p_safe:.3f}")

<details>
<summary>✅ <b>Solution</b></summary>

```python
profile = dict(tenure_months=3, monthly_charges=95.0, support_tickets=4)

p_risky = churn_probability(**profile, contract="month-to-month")
p_safe = churn_probability(**profile, contract="two-year")

assert 0.0 <= p_risky <= 1.0 and 0.0 <= p_safe <= 1.0
assert p_risky > p_safe, "month-to-month should out-risk two-year"
print(f"month-to-month {p_risky:.3f}  >  two-year {p_safe:.3f}")
```

Those two asserts are `example-app/scoring/tests.py` (`test_month_to_month_is_riskier`, `test_probability_in_range`) written as notebook lines — §7 folds them into the full suite. Note what you *didn't* need to know: that there's a `StandardScaler`, a one-hot encoding, or a DataFrame inside. The seam hides the ML from the app — which is exactly what lets the example app swap its stand-in for this pipeline without touching a single view.
</details>

## 4. The JSON API — `POST /api/score/`

First serving surface: robots. Scripts, cron jobs, other services — the callers from Modules 7 and 8 — speak JSON, not HTML forms. The view below is `api_score` from `example-app/scoring/views.py`, now calling our *trained* model through the seam: parse JSON → validate/coerce → predict → **log the `Prediction` row** → return JSON. Bad input never reaches the model — it gets a clean `400` with a reason instead of a stack trace.

> ⚠️ **What `@csrf_exempt` really does.** CSRF protection defends *browser* users: since browsers attach session cookies automatically, a malicious page could POST to ChurnScope *as you* — so Django requires every POST to carry a token proving it came from a page Django itself rendered. A JSON API called by scripts has **no session cookie to abuse and no form to plant a token in** — the shield can't work there, so `@csrf_exempt` switches it off *for this one view*. The honest cost: the endpoint is now open, so real deployments pair it with **token authentication** (DRF or Django Ninja, per the chapter) instead of cookies. Never sprinkle `@csrf_exempt` on *form* views to make an error go away — §5 shows the shield doing its job.

In [ ]:
if HAS_DJANGO:
    import json

    from django.http import JsonResponse
    from django.views.decorators.csrf import csrf_exempt
    from django.views.decorators.http import require_POST

    THRESHOLD = 0.5

    @csrf_exempt          # stateless JSON API — see the ⚠️ above
    @require_POST         # GET /api/score/ makes no sense → automatic 405
    def api_score(request):
        """POST JSON → {"probability": float, "will_churn": bool} — and log it."""
        try:
            payload = json.loads(request.body or "{}")
            data = {
                "tenure_months": int(payload["tenure_months"]),
                "monthly_charges": float(payload["monthly_charges"]),
                "support_tickets": int(payload["support_tickets"]),
                "contract": str(payload["contract"]),
            }
        except (KeyError, ValueError, TypeError) as exc:
            return JsonResponse({"error": f"invalid request: {exc}"}, status=400)

        p = churn_probability(**data)              # the seam, again
        will_churn = p >= THRESHOLD
        Prediction.objects.create(probability=round(p, 4), will_churn=will_churn, **data)
        return JsonResponse({"probability": round(p, 4), "will_churn": will_churn})

    print("api_score defined — a function until a URL points at it.")
else:
    print("Django not installed — skipping.")

Route it — the URLconf is a module we assemble by hand and plant in `sys.modules` (Lab 1 §6's trick), matching the `ROOT_URLCONF = "notebook_urls"` promise from §2. Then call it with the test **`Client`**: a browser without the browser, pushing a real request through URLconf → view → model → ORM.

In [ ]:
if HAS_DJANGO:
    import types

    from django.test import Client
    from django.urls import clear_url_caches, include, path

    urlconf = types.ModuleType("notebook_urls")
    urlconf.urlpatterns = [
        path("api/score/", api_score, name="api_score"),
    ]
    sys.modules["notebook_urls"] = urlconf
    clear_url_caches()          # Django caches the resolver — flush after edits

    client = Client()

    CUSTOMER = {"tenure_months": 3, "monthly_charges": 95,
                "support_tickets": 4, "contract": "month-to-month"}

    resp = client.post("/api/score/", data=json.dumps(CUSTOMER),
                       content_type="application/json")
    print("POST /api/score/ →", resp.status_code, resp.json())
    print("rows logged so far:", Prediction.objects.count())
else:
    print("Django not installed — skipping.")

The same `curl` from the chapter would get the same JSON. Now the unhappy paths — a production API is *defined* by how it fails. Watch three failures produce three clean, correct responses (the red `Bad Request` / `Method Not Allowed` lines are Django's request logger agreeing with us):

In [ ]:
if HAS_DJANGO:
    missing = client.post("/api/score/", data='{"monthly_charges": 95}',
                          content_type="application/json")
    print("missing fields      →", missing.status_code, missing.json())

    not_json = client.post("/api/score/", data="tenure=3&contract=gold",
                           content_type="application/json")
    print("body isn't JSON     →", not_json.status_code, not_json.json())

    wrong_method = client.get("/api/score/")
    print("GET instead of POST →", wrong_method.status_code, "(require_POST at work)")

    # And the ⚠️ from above, demonstrated: a CSRF-strict client (a stand-in for a
    # real browser without a token) still passes, because of @csrf_exempt.
    strict = Client(enforce_csrf_checks=True)
    r = strict.post("/api/score/", data=json.dumps(CUSTOMER),
                    content_type="application/json")
    print("CSRF-strict POST    →", r.status_code, "(csrf_exempt lets robots in)")
else:
    print("Django not installed — skipping.")

---

### ✋ Quick exercise (~2 min) — The ledger never lies

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

"Log every prediction" is a *claim* — audit it. POST a loyal customer (`tenure_months=48, monthly_charges=60, support_tickets=0, contract="two-year"`) to `/api/score/` and assert the whole contract: status `200`; `probability` in `[0, 1]`; `will_churn` consistent with `probability >= THRESHOLD`; **exactly one** new `Prediction` row (count it before and after); and the newest row (`order_by("-pk").first()`) is your two-year customer.

In [ ]:
# ✍️ Your turn 👇
loyal = {"tenure_months": 48, "monthly_charges": 60,
         "support_tickets": 0, "contract": "two-year"}

# before = Prediction.objects.count()
# resp = client.post("/api/score/", data=json.dumps(loyal), content_type="application/json")
resp = ...

# body = resp.json()
# assert resp.status_code == 200
# assert 0.0 <= body["probability"] <= 1.0
# assert body["will_churn"] == (body["probability"] >= THRESHOLD)
# assert Prediction.objects.count() == before + 1
# newest = Prediction.objects.order_by("-pk").first()
# assert newest.contract == "two-year"
# print("logged:", newest)

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    loyal = {"tenure_months": 48, "monthly_charges": 60,
             "support_tickets": 0, "contract": "two-year"}

    before = Prediction.objects.count()
    resp = client.post("/api/score/", data=json.dumps(loyal),
                       content_type="application/json")
    body = resp.json()

    assert resp.status_code == 200
    assert 0.0 <= body["probability"] <= 1.0
    assert body["will_churn"] == (body["probability"] >= THRESHOLD)
    assert Prediction.objects.count() == before + 1
    newest = Prediction.objects.order_by("-pk").first()
    assert newest.contract == "two-year"
    print("logged:", newest, "| rows now:", Prediction.objects.count())
else:
    print("Django not installed — skipping.")
```

A 48-month two-year customer with zero tickets scores near zero — and the row proves the API kept its side-effect promise. Two details worth stealing: the **before/after count** (never assert absolute counts — other cells also log rows), and `order_by("-pk")` for "the row just inserted" (the model's `-created_at` ordering can tie when two rows land in the same clock tick; primary keys never tie).
</details>

## 5. The HTML form — the same seam, for humans

Second surface: people. The `index` view below is `example-app/scoring/views.py` line for line (the example app mounts it at the site root `/`; we mount it at `/score/`): GET shows the form, POST validates through **`ChurnForm`**, calls `churn_probability`, **logs the row**, and re-renders with the verdict. Note what it has in common with `api_score` — validate → predict → log → respond — and what differs: the form does the validating, HTML does the talking, and **CSRF stays on**, because this *is* a browser flow.

In [ ]:
if HAS_DJANGO:
    from django.shortcuts import render

    TEMPLATE_REGISTRY["scoring/form.html"] = """\
<h1>ChurnScope — score a customer</h1>
<form method="post">
  {% csrf_token %}
  {{ form.as_p }}
  <button type="submit">Score</button>
</form>
{% if result %}<p id="result">Churn probability: <b>{{ result.probability|floatformat:2 }}</b> —
{% if result.will_churn %}likely to CHURN{% else %}likely to stay{% endif %}
<small>(model {{ result.model_version }})</small></p>{% endif %}"""

    def index(request):
        """The churn form: GET shows it, POST scores + logs + shows the result."""
        result = None
        if request.method == "POST":
            form = ChurnForm(request.POST)
            if form.is_valid():
                data = form.cleaned_data
                p = churn_probability(**data)
                will_churn = p >= THRESHOLD
                Prediction.objects.create(probability=round(p, 4),
                                          will_churn=will_churn, **data)
                result = {"probability": p, "will_churn": will_churn,
                          "model_version": MODEL_VERSION}
        else:
            form = ChurnForm()
        return render(request, "scoring/form.html",
                      {"form": form, "result": result, "threshold": THRESHOLD})

    urlconf.urlpatterns = [
        path("score/", index, name="index"),
        path("api/score/", api_score, name="api_score"),
    ]
    clear_url_caches()

    page = client.get("/score/")
    print("GET /score/ →", page.status_code, "(the empty form)")

    before = Prediction.objects.count()
    resp = client.post("/score/", {"tenure_months": 2, "monthly_charges": 99.5,
                                   "support_tickets": 4, "contract": "month-to-month"})
    print("POST /score/ →", resp.status_code)
    for line in resp.content.decode().splitlines():
        if 'id="result"' in line or "likely to" in line or "model " in line:
            print("   ", line.strip())
    print(f"rows: {before} → {Prediction.objects.count()}   ← the view logged it")
else:
    print("Django not installed — skipping.")

And the promised counter-demonstration: the **same CSRF-strict client** that sailed through the API gets stopped cold by the form view, because a POST without the `{% csrf_token %}` stamp is exactly what a cross-site forgery looks like. (The default test `Client` politely carries a skip flag, which is why our earlier POST worked without a token — a real browser gets no such courtesy.)

In [ ]:
if HAS_DJANGO:
    r = strict.post("/score/", {"tenure_months": 2, "monthly_charges": 99.5,
                                "support_tickets": 4, "contract": "month-to-month"})
    print("CSRF-strict POST /score/ →", r.status_code, "— the shield works")
    print("fix for browsers: keep {% csrf_token %} in the form (we did);")
    print("fix for robots  : don't use cookie auth at all → tokens (DRF/Ninja)")
else:
    print("Django not installed — skipping.")

## 6. Auth — the last battery, and the protected history page

Every score so far left a `Prediction` row behind; the **history page** is where humans read that ledger — and it must not be public. [auth-and-history.md](auth-and-history.md)'s whole pitch: you write almost no auth code, because `django.contrib.auth` (installed in §2) ships users, password hashing, sessions, and complete login/logout views. You add exactly three things — mount `django.contrib.auth.urls`, set the three redirect settings (§2 did), give the login view a `registration/login.html` template — and then protect any view with **one decorator**:

- `@login_required` bounces strangers to `LOGIN_URL` with `?next=` so they land back where they were heading;
- the view itself stays four lines: paginate the queryset, render. Newest-first ordering comes free from the model's `Meta.ordering`.

We also add a tiny `whoami` view, because the honest way to understand auth is to watch **`request.user`** change: `AuthenticationMiddleware` reads the session cookie and attaches either a real `User` or `AnonymousUser` to every request — *that attribute is what "logged in" means.*

In [ ]:
if HAS_DJANGO:
    from django.contrib.auth.decorators import login_required
    from django.core.paginator import Paginator

    TEMPLATE_REGISTRY["scoring/history.html"] = """\
<h1>Prediction history</h1>
<p>page {{ page_obj.number }} of {{ page_obj.paginator.num_pages }} — signed in as <b>{{ user.username }}</b></p>
<table>
  <tr><th>#</th><th>when</th><th>contract</th><th>p</th><th>verdict</th></tr>
{% for p in page_obj %}  <tr><td>{{ forloop.counter }}</td><td>{{ p.created_at|date:"Y-m-d H:i" }}</td><td>{{ p.contract }}</td><td>{{ p.probability|floatformat:2 }}</td><td>{% if p.will_churn %}<b>churn</b>{% else %}stay{% endif %}</td></tr>
{% endfor %}</table>"""

    TEMPLATE_REGISTRY["registration/login.html"] = """\
<h1>Sign in to ChurnScope</h1>
<form method="post">
  {% csrf_token %}
  {{ form.as_p }}
  <button type="submit">Log in</button>
</form>"""

    @login_required
    def history(request):
        """Paginated prediction log, newest first (per the model's Meta.ordering)."""
        paginator = Paginator(Prediction.objects.all(), per_page=10)
        page_obj = paginator.get_page(request.GET.get("page"))
        return render(request, "scoring/history.html", {"page_obj": page_obj})

    def whoami(request):
        """Who does Django think is asking? request.user always knows."""
        return JsonResponse({"user": str(request.user),
                             "is_authenticated": request.user.is_authenticated})

    urlconf.urlpatterns = [
        path("score/", index, name="index"),
        path("api/score/", api_score, name="api_score"),
        path("history/", history, name="history"),
        path("whoami/", whoami, name="whoami"),
        # Django's complete login/logout/password machinery — one include:
        path("accounts/", include("django.contrib.auth.urls")),
    ]
    clear_url_caches()

    print("Routes:", ", ".join(f"/{p.pattern}" for p in urlconf.urlpatterns))
else:
    print("Django not installed — skipping.")

**Act one: a stranger knocks.** Anonymous request → `302` to the login page, with `?next=` remembering the destination — and the login page itself already works, rendered by Django's own `LoginView` through our nine-line template:

In [ ]:
if HAS_DJANGO:
    print("whoami, anonymous  :", client.get("/whoami/").json())

    bounced = client.get("/history/")
    print("GET /history/      :", bounced.status_code, "→", bounced.headers["Location"])

    login_page = client.get(bounced.headers["Location"])
    print("GET the login page :", login_page.status_code, "— Django's own LoginView")
    print("\n".join(login_page.content.decode().splitlines()[:4]))
else:
    print("Django not installed — skipping.")

**Act two: give someone a key.** `User.objects.create_user(...)` — and only ever that.

> ⚠️ **`create_user`, never `User(password=...)`.** The constructor stores whatever you pass — *verbatim, in plaintext*. `create_user` runs the password through Django's salted PBKDF2 hasher; look at the stored value below and count what an attacker who steals the database learns: nothing. (Same rule via `user.set_password(...)` when changing one.)

In [ ]:
if HAS_DJANGO:
    from django.contrib.auth.models import User

    analyst = User.objects.filter(username="analyst").first()   # idempotent re-runs
    if analyst is None:
        analyst = User.objects.create_user("analyst", password="pass1234")

    print("user   :", analyst.username)
    print("stored :", analyst.password[:45] + "…")
    print("         ↑ algorithm, iterations, salt, hash — never the password")
else:
    print("Django not installed — skipping.")

**Act three: walk in.** In a browser the analyst would type the password into that login form; the test client's working path is **`force_login(user)`** — it plants a valid session for the user directly, skipping the password check and hashers. That's the standard fast path in Django's own test suites: *auth flows you test once, protected views you test everywhere*, and `force_login` keeps "everywhere" fast. The full form flow is one POST, for reference — it works here too, it's just not the path we'll lean on:

```python
# the browser-faithful path (reference — force_login below is the working path):
fresh = Client()
resp = fresh.post("/accounts/login/",
                  {"username": "analyst", "password": "pass1234"}, follow=True)
# Django checks the password, creates the session, then follows the redirect
# to LOGIN_REDIRECT_URL — so resp is the /history/ page, already logged in.
```

In [ ]:
if HAS_DJANGO:
    client.force_login(analyst)          # a valid session, no password dance

    resp = client.get("/history/")
    print("GET /history/ →", resp.status_code, "  (302 → login → 200: the full arc)\n")
    print("\n".join(resp.content.decode().splitlines()[:8]))
    print("…")
    print("\nwhoami, logged in:", client.get("/whoami/").json())
else:
    print("Django not installed — skipping.")

---

### ✋ Quick exercise (~2 min) — A second pair of eyes

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The analyst isn't the only one with a key. Create a user `"intern"` (idempotently — re-running the cell must not crash), log them in with `force_login` **on a brand-new `Client`**, and check: `/history/` returns `200` and still contains `"month-to-month"`, and `/whoami/` names the intern. Then answer the real question: whose predictions is the intern looking at — and is that a feature or a bug?

In [ ]:
# ✍️ Your turn 👇
# intern = User.objects.filter(username="intern").first()
# if intern is None:
#     intern = ...
intern = ...

# second = Client()
# second.force_login(intern)
# resp = second.get("/history/")
# assert resp.status_code == 200 and "month-to-month" in resp.content.decode()
# print(second.get("/whoami/").json())

<details>
<summary>✅ <b>Solution</b></summary>

```python
if HAS_DJANGO:
    intern = User.objects.filter(username="intern").first()
    if intern is None:
        intern = User.objects.create_user("intern", password="intern-pw-42")

    second = Client()
    second.force_login(intern)

    resp = second.get("/history/")
    assert resp.status_code == 200
    assert "month-to-month" in resp.content.decode()
    print(second.get("/whoami/").json())
    print("→ the intern sees EVERY prediction, including the analyst's")
else:
    print("Django not installed — skipping.")
```

Both, honestly. `@login_required` answers *"are you anyone?"*, not *"are you allowed to see this row?"* — the history is one global ledger, which is fine for a small team's audit trail and wrong the moment customers log in. Authorisation (per-user filtering, permissions) is a separate layer on top of authentication — Stretch exercise territory, and the reason `Prediction` would grow a `user` foreign key in a multi-tenant ChurnScope.
</details>